In [1]:
# Import all functions from the required modules
from pop_functions_module import *
print("Successfully loaded population functions")

Successfully loaded population functions


In [2]:
# Paths to data files
path_contours = "data/1-processed-data/CONTOURS-IRIS"
path_iris = "data/1-processed-data/INSEE/donnees_iris.txt"
path_proj = "data/1-processed-data/INSEE/donnees_projections.txt"
path_depart = "data/1-processed-data/INSEE/num_depart.csv"
path_age_femmes = "data/1-processed-data/INSEE/donnees_age_femmes.csv"
path_age_hommes = "data/1-processed-data/INSEE/donnees_age_hommes.csv"
path_mortalite_hf = "data/1-processed-data/INSEE/donnees_deces_hf.csv"
#path_pop_density = "data/1-processed-data/INSEE/donnees_dens.csv"

# EXPORT : paths and titles for exported data
path = "data/2-output-data"
path_fichier_shp = "data/2-output-data/donnees_shp"
path_fichier_shp_1 = "data/2-output-data/donnees_shp_1"
path_fichier_shp_2 = "data/2-output-data/donnees_shp_2"
path_fichier_shp_3 = "data/2-output-data/donnees_shp_3"
title_shp = "donnees_insee_iris"
title_shp_1 = "donnees_insee_iris_toutage_1"
title_shp_2 = "donnees_insee_iris_toutage_2"
title_shp_3 = "donnees_insee_iris_toutage_3"

In [3]:
donnees_geom = geometries(path_iris, path_depart, path_contours)
print("Top 2 rows of donnees_geom:")
print(donnees_geom.head())
age_hf = age_nat(path_age_femmes, path_age_hommes)
print("Top 2 rows of age_hf:")
print(age_hf.head())
perc_hf = decomposition_age(age_hf)
print("All columns of perc_hf:")
print(perc_hf.columns)
print("Top 2 rows of perc_hf:")
print(perc_hf.head())
pd.set_option('display.max_columns', None)  # Ensures all columns are shown
donnees_proj = recense(path_proj, perc_hf)
print("Top 2 rows of donnees_proj:")
print(donnees_proj.head())
donnees_insee = desagreg(donnees_geom, donnees_proj)
print("Top 2 rows of donnees_insee:")
print(donnees_insee.head())
donnees_dens = dens(donnees_insee)
print("Top 2 rows of donnees_dens:")
print(donnees_dens.head())
donnees_finales = mortalite(donnees_dens, path_mortalite_hf)
print("Top 2 rows of donnees_finales:")
print(donnees_finales.head())
donnees_filtrees = create_donnees_shp(donnees_finales)

The INSEE data is extracted at the IRIS level.
Top 2 rows of donnees_geom:
  dep_cod        dep_name  region_cod       region_name com_cod  com_name  \
0      72          Sarthe        52.0  Pays de la Loire   72191     Mayet   
1      77  Seine-et-Marne        11.0     Ile-de-France   77248   Lesches   
2      51           Marne        44.0         Grand Est   51426      Péas   
3      81            Tarn        76.0         Occitanie   81199    Padiès   
4      59            Nord        32.0   Hauts-de-France   59225  Feignies   

    iris_cod iris_name  POP_2019_IRIS_TotAge  \
0  721910000     Mayet                3120.0   
1  772480000   Lesches                 763.0   
2  514260000      Péas                  66.0   
3  811990000    Padiès                 189.0   
4  592250102       Sud                2479.0   

                                            geometry       aire_m2  
0  POLYGON ((498083.5 6747517.4, 498128 6747467.1...  5.418755e+07  
1  POLYGON ((685753.1 6868612.9, 68

In [7]:
# Export donnees_filtrees filtered to depcod=75 to CSV in the same output directory as other CSV files
csv_path = f"data/1-processed-data/INSEE/mort_projection_Paris.csv"
df_75 = donnees_filtrees.copy()
df_75["depcod"] = df_75["depcod"].astype(str).str.zfill(2)
df_75 = df_75[df_75["depcod"] == "75"]
df_75.to_csv(csv_path, index=False, encoding="utf-8")
print(f"Saved depcod=75 filtered donnees_filtrees to: {csv_path}")

Saved depcod=75 filtered donnees_filtrees to: data/1-processed-data/INSEE/mort_projection_Paris.csv


In [6]:
donnees_shp = export_pollution(donnees_filtrees, donnees_geom)
gdf = gpd.GeoDataFrame(donnees_shp, geometry=donnees_geom.geometry)

# Convert other dataframes to GeoDataFrames
donnees_shp_1 = donnees_filtrees.query("age >= 0 & age <= 55").loc[:, [
                                                                          "iriscod", "irisname", "comcod", "comname",
                                                                          "depcod", "depname", "regcod", "regname",
                                                                          "age",
                                                                          "pop2019", "pop2030", "pop2050", "mort2019",
                                                                          "mort2030", "mort2050"
                                                                      ]]
gdf_1 = gpd.GeoDataFrame(donnees_shp_1, geometry=donnees_geom.geometry)

donnees_shp_2 = donnees_filtrees.query("age >= 56 & age <= 87").loc[:, [
                                                                           "iriscod", "irisname", "comcod", "comname",
                                                                           "depcod", "depname", "regcod", "regname",
                                                                           "age",
                                                                           "pop2019", "pop2030", "pop2050", "mort2019",
                                                                           "mort2030", "mort2050"
                                                                       ]]
gdf_2 = gpd.GeoDataFrame(donnees_shp_2, geometry=donnees_geom.geometry)

donnees_shp_3 = donnees_filtrees.query("age >= 88 & age <= 99").loc[:, [
                                                                           "iriscod", "irisname", "comcod", "comname",
                                                                           "depcod", "depname", "regcod", "regname",
                                                                           "age",
                                                                           "pop2019", "pop2030", "pop2050", "mort2019",
                                                                           "mort2030", "mort2050"
                                                                       ]]
gdf_3 = gpd.GeoDataFrame(donnees_shp_3, geometry=donnees_geom.geometry)

export_data_shp(gdf, path_fichier_shp, title_shp)
export_data_shp(gdf_1, path_fichier_shp_1, title_shp_1)
export_data_shp(gdf_2, path_fichier_shp_2, title_shp_2)
export_data_shp(gdf_3, path_fichier_shp_3, title_shp_3)

# Functions for exportation
title_csv = "my_data_file"
#export_data_csv(donnees_finales, path, title_csv)
#export_data_shp(donnees_indic, path_fichier_shp, title_shp)
#donnees_exportees = gpd.read_file(os.path.join(path_fichier_shp_1, f"{title_shp_1}.shp"))
#print(donnees_exportees.head())


Shapefile written to: data/2-output-data/donnees_shp\donnees_insee_iris.shp
Shapefile written to: data/2-output-data/donnees_shp_1\donnees_insee_iris_toutage_1.shp
Shapefile written to: data/2-output-data/donnees_shp_2\donnees_insee_iris_toutage_2.shp
Shapefile written to: data/2-output-data/donnees_shp_3\donnees_insee_iris_toutage_3.shp


In [ ]:
# Functions to test
annee = 2030
dep_num = "75"
iris_num = "010040102"
n = 35
national_vivant_test(donnees_finales, annee)
national_mort_test(donnees_finales, annee)
departemental_test(donnees_finales, dep_num, annee)
iris_test(donnees_finales)
dep_vs_iris_test(donnees_finales)
pyramide_iris(donnees_insee, iris_num, annee)
pyramide_dep(donnees_insee, dep_num, annee)
pyramide_nat(donnees_insee, annee)

In [ ]:
#Code to classify data into Urban vs Rural
# Step 1: Aggregate IRIS-level data to commune level (by "comcod")
commune_level_data = (
    donnees_exportees.groupby("comcod")
    .agg(
        pop2019=("pop2019", "sum"),
        pop2030=("pop2030", "sum"),
        pop2050=("pop2019", "sum"),
        area=("geometry", lambda x: x.union_all().area),  # Compute total area by unioning geometries
        geometry=("geometry", lambda x: x.union_all())  # Combine geometries to create commune-level geometry
    )
    .reset_index()
)

# Step 2: Calculate population density at the commune level
commune_level_data["pop_density"] = commune_level_data["pop2030"] / (commune_level_data["area"] / 1e6)

# Debugging: Display the first few rows of commune-level data
print("Head of commune-level data:")
print(commune_level_data.head())

# Step 3: Classify communes into "Urban" or "Rural" clusters
def classify_area_clusters(row):
    if row["pop_density"] > 1500:
        if row["pop2019"] >= 50000 or (5000 <= row["pop2019"] < 50000):
            return "Urban"  # Merge all Dense Urban levels into Urban
    elif row["pop_density"] > 300 and row["pop2019"] >= 5000:
        return "Urban"  # Include Semi-Dense Urban in Urban category
    else:
        return "Rural"  # Merge Rural and Other together

# Apply classification to the commune-level data
commune_level_data["area_cluster"] = commune_level_data.apply(classify_area_clusters, axis=1)

# Step 4: Debugging - Count clusters after classification
print(f"Clusters at the commune level:\n{commune_level_data['area_cluster'].value_counts()}")

# Step 5: Disaggregate classification back to the IRIS level
# Merge commune-level results (area_cluster, pop_density) back to the IRIS-level data
donnees_exportees_transformed = donnees_exportees.merge(
    commune_level_data[["comcod", "area_cluster", "pop_density"]],
    on="comcod",  # Match by the commune code
    how="left"  # Retain all IRIS-level data
)

# Step 6: Debugging - Display the first few rows of the disaggregated data
print("Head of IRIS-level data with disaggregated classification:")
print(donnees_exportees_transformed.head())

In [ ]:
#Distribute IRIS based disaggregation/distribution using donnees_merged file of 2019
import pandas as pd
import numpy as np
import os

# --- Read mortality file (XLSX only) ---
mort_file = 'data/1-processed-data/INSEE/morta30_1619_envoye.xlsx'
mort_df = pd.read_excel(mort_file, dtype=str, engine='openpyxl')
mort_df.columns = mort_df.columns.str.strip()
for col in mort_df.select_dtypes(include='object'):
    mort_df[col] = mort_df[col].strip() if isinstance(mort_df[col], str) else mort_df[col]

# Standardize columns
if 'insee2' in mort_df.columns:
    mort_df = mort_df.rename(columns={'insee2': 'comcod'})
    mort_df['comcod'] = mort_df['comcod'].str.upper().str.zfill(5)
else:
    raise ValueError("Column 'insee2' not found in Excel file.")

if 'morta_a30_moy' in mort_df.columns:
    mort_df['morta_a30_moy'] = pd.to_numeric(mort_df['morta_a30_moy'].str.replace(' ', ''), errors='coerce').fillna(0.0)
else:
    raise ValueError("Column 'morta_a30_moy' not found in Excel file.")

# Apply arrondissement mapping
#mort_df['comcod'] = mort_df['comcod'].map(lambda x: arrondissement_mapping.get(x, x))

# --- Read population file (optional) ---
population_file = "data/Morbidity data check/population_data.xlsx"
if os.path.exists(population_file):
    pop_comm = pd.read_excel(population_file, dtype={"insee": str}, engine="openpyxl")
    pop_comm["comcod"] = pop_comm["insee"].str.upper().str.zfill(5)
    pop_comm["pop30p"] = pd.to_numeric(pop_comm["pop30p"], errors="coerce").fillna(0)
else:
    print(f"Population file not found: {population_file} (continuing with IRIS population only)")

# --- Prepare IRIS × age population ---
donnees_merged_df = donnees_merged.copy()
donnees_merged_df["comcod"] = donnees_merged_df["comcod"].str.upper().str.zfill(5)
donnees_merged_df["comcod"] = donnees_merged_df["comcod"].map(lambda x: arrondissement_mapping.get(x, x))
donnees_merged_df["iriscod"] = donnees_merged_df["iriscod"].astype(str).str.zfill(5)
donnees_merged_df["depcod"] = donnees_merged_df["depcod"].astype(str).str.zfill(2)
donnees_merged_df["regcod"] = donnees_merged_df["regcod"].astype(str).str.zfill(2)

# --- Step 1: Aggregate to commune × age ---
comm_age_pop = donnees_merged_df.groupby(["comcod", "age"], as_index=False)["pop2019"].sum()
comm_age_pop = comm_age_pop[comm_age_pop["age"].between(30, 99)]

# --- Remove communes with zero population ---
total_pop_by_comm = comm_age_pop.groupby("comcod")["pop2019"].transform("sum")
comm_age_pop = comm_age_pop[total_pop_by_comm > 0]

# --- Step 2: Merge with commune mortality and pop30p ---
# Merge pop_comm to get pop30p for each commune
commune_df = mort_df.merge(pop_comm[["comcod", "pop30p"]], on="comcod", how="left")
comm_mort_age = comm_age_pop.merge(
    commune_df[["comcod", "morta_a30_moy", "pop30p"]],
    on="comcod",
    how="left"
)

# --- Fill missing mortality using dep/reg codes ---
missing_mask = comm_mort_age["morta_a30_moy"].isna()
if missing_mask.any():
    commune_df["depcod"] = commune_df["comcod"].str[:2]
    commune_df["regcod"] = commune_df["comcod"].str[:2]
    dep_lookup = commune_df.groupby("depcod")["morta_a30_moy"].first()
    reg_lookup = commune_df.groupby("regcod")["morta_a30_moy"].first()

    comm_mort_age.loc[missing_mask, "depcod"] = comm_mort_age.loc[missing_mask, "comcod"].str[:2]
    comm_mort_age.loc[missing_mask, "morta_a30_moy"] = comm_mort_age.loc[missing_mask, "depcod"].map(dep_lookup)

    still_missing_mask = comm_mort_age["morta_a30_moy"].isna()
    comm_mort_age.loc[still_missing_mask, "regcod"] = comm_mort_age.loc[still_missing_mask, "comcod"].str[:2]
    comm_mort_age.loc[still_missing_mask, "morta_a30_moy"] = comm_mort_age.loc[still_missing_mask, "regcod"].map(reg_lookup)

# --- Step 3: Compute mortality and pop30p per commune × age ---
total_pop_by_comm = comm_mort_age.groupby("comcod")["pop2019"].transform("sum")
# Distribute pop30p to each age proportionally
comm_mort_age["pop30p_age"] = np.where(
    total_pop_by_comm > 0,
    (comm_mort_age["pop2019"] / total_pop_by_comm) * comm_mort_age["pop30p"],
    0
)
comm_mort_age["mort_age"] = np.where(
    total_pop_by_comm > 0,
    (comm_mort_age["pop2019"] / total_pop_by_comm) * comm_mort_age["morta_a30_moy"],
    0
)

# --- Step 4: Distribute to IRIS × age ---
iris_mort_age = donnees_merged_df.merge(
    comm_mort_age[["comcod", "age", "mort_age", "pop30p_age"]],
    on=["comcod", "age"],
    how="left"
)

# Population-weighted mortality and pop30p allocation at IRIS level (within the same commune × age)
total_pop_by_comm_age = iris_mort_age.groupby(["comcod", "age"])["pop2019"].transform("sum")
iris_mort_age["mort_iris_age"] = np.where(
    total_pop_by_comm_age > 0,
    (iris_mort_age["pop2019"] / total_pop_by_comm_age) * iris_mort_age["mort_age"],
    0
)
iris_mort_age["pop30p_iris_age"] = np.where(
    total_pop_by_comm_age > 0,
    (iris_mort_age["pop2019"] / total_pop_by_comm_age) * iris_mort_age["pop30p_age"],
    0
)

# --- Step 5: Final IRIS-level output ---
donnees_merged_iris = iris_mort_age[["iriscod", "age", "pop2019", "pop30p_iris_age", "mort_iris_age"]].rename(
    columns={"mort_iris_age": "mortality", "pop30p_iris_age": "pop30p"}
)

# --- Sanity check ---
input_sum = commune_df["morta_a30_moy"].sum()
output_sum = donnees_merged_iris["mortality"].sum()
input_pop30p = commune_df["pop30p"].sum()
output_pop30p = donnees_merged_iris["pop30p"].sum()
print("Total commune mortality (input):", input_sum)
print("Total allocated to IRIS:", output_sum)
print("Total commune pop30p (input):", input_pop30p)
print("Total allocated pop30p to IRIS:", output_pop30p)
if not np.allclose(input_sum, output_sum):
    print("WARNING: Mismatch in distributed mortality sums!")
if not np.allclose(input_pop30p, output_pop30p):
    print("WARNING: Mismatch in distributed pop30p sums!")

donnees_merged_iris.head()